# Notebook 20. Large-Scale Pressure Context for the Cleaned Clusters

This notebook replaces the earlier Siberian-High-only version with a broader pressure-context analysis.

**Main question:** do cleaned `Cluster 1` and cleaned `Cluster 2` occur under different large-scale surface-pressure configurations, especially in relation to the Eastern Siberian High, the Aleutian Low, and southward ridge extension toward East Asia?

Primary goals:

- use the cleaned `k = 2` labels from `Notebook 15`
- use `mean sea-level pressure` (`msl`) anomaly instead of low-level pressure-level fields
- identify the **Eastern Siberian High center** inside a broad Siberian search box
- identify the **Aleutian Low center** inside a broad North Pacific search box
- quantify **ridge-extension strength** in a separate East Asia / East China Sea box
- compare those metrics across the cleaned clusters

Secondary overlay:

- if `Notebook 18` quartile selection is available, compare these pressure-context metrics across the `lower / middle / upper` subsets of cleaned `Cluster 1`
- keep all of this as a diagnostic, not as a new clustering variable


In [2]:
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/angelicasophyaramirez-blip/JPCZcatalogcolab.git"
BRANCH = "main"
REPO_DIR = "/content/JPCZcatalog"
FORCE_REFRESH_REPO = False
PERSIST_OUTPUTS_TO_DRIVE = True
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/JPCZcatalog_outputs"

if PERSIST_OUTPUTS_TO_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
    print("Persistent output dir:", DRIVE_OUTPUT_DIR)

os.chdir("/content")

if FORCE_REFRESH_REPO and os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
    print("Removed existing repo clone:", REPO_DIR)

if not os.path.exists(REPO_DIR):
    proc = subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR],
        text=True,
        capture_output=True,
    )
    print(proc.stdout)
    print(proc.stderr)
    if proc.returncode != 0:
        raise RuntimeError(f"git clone failed:\n{proc.stderr}")

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements-colab.txt"],
        check=True,
    )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR],
        check=True,
    )
else:
    print("Using existing repo clone:", REPO_DIR)

os.chdir(REPO_DIR)
src_dir = os.path.join(REPO_DIR, "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

print("Working directory:", os.getcwd())


Mounted at /content/drive
Persistent output dir: /content/drive/MyDrive/JPCZcatalog_outputs

Cloning into '/content/JPCZcatalog'...

Working directory: /content/JPCZcatalog


In [ ]:
from pathlib import Path
import shutil

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import numpy as np
import pandas as pd
import xarray as xr

from jpcz_catalog.analysis import add_japan_local_time_columns
from jpcz_catalog.config import BoundingBox
from jpcz_catalog.diagnostics import load_snapshot
from jpcz_catalog.era5 import open_arco_era5

CLEANED_RUN_EXPORT_DIR = Path("outputs/verification/objective_subtype_low_level_cleaned_sensitivity")
QUARTILE_EXPORT_DIR = Path("outputs/verification/objective_subtype_cleaned_cluster_quartiles")
PRESSURE_CONTEXT_EXPORT_DIR = Path("outputs/verification/objective_subtype_large_scale_pressure_context")
PRESSURE_CONTEXT_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR = Path("outputs/verification/objective_subtype_large_scale_pressure_context_plots")
PLOT_DIR.mkdir(parents=True, exist_ok=True)

CLEANED_CLUSTERED_EVENTS_PATH = CLEANED_RUN_EXPORT_DIR / "clustered_events_cleaned_low_level_k2_k3_k4.csv"
NOTEBOOK15_SOLUTION_SUMMARY_PATH = CLEANED_RUN_EXPORT_DIR / "cleaned_low_level_solution_summary.csv"
QUARTILE_SELECTION_PATH = QUARTILE_EXPORT_DIR / "cleaned_k2_cluster1_quartile_selection.csv"

MSL_CLIMATOLOGY_PATH = PRESSURE_CONTEXT_EXPORT_DIR / "msl_ndjf_monthly_climatology_eastasia_npacific_domain.nc"
EVENT_STACK_PATH = PRESSURE_CONTEXT_EXPORT_DIR / "cleaned_k2_large_scale_pressure_event_anomaly_stack.nc"
EVENT_STACK_PARTIAL_PATH = PRESSURE_CONTEXT_EXPORT_DIR / "cleaned_k2_large_scale_pressure_event_anomaly_stack_partial.nc"
EVENT_METRICS_PATH = PRESSURE_CONTEXT_EXPORT_DIR / "cleaned_k2_large_scale_pressure_event_metrics.csv"
EVENT_METRICS_PARTIAL_PATH = PRESSURE_CONTEXT_EXPORT_DIR / "cleaned_k2_large_scale_pressure_event_metrics_partial.csv"
CLUSTER_COMPOSITE_PATH = PRESSURE_CONTEXT_EXPORT_DIR / "cleaned_k2_large_scale_pressure_cluster_composites.nc"
CLUSTER_SUMMARY_PATH = PRESSURE_CONTEXT_EXPORT_DIR / "cleaned_k2_large_scale_pressure_cluster_summary.csv"
POSITION_CATEGORY_PATH = PRESSURE_CONTEXT_EXPORT_DIR / "cleaned_k2_large_scale_pressure_position_category_summary.csv"
QUARTILE_PRESSURE_SUMMARY_PATH = PRESSURE_CONTEXT_EXPORT_DIR / "cleaned_k2_cluster1_large_scale_pressure_quartile_summary.csv"
PLOT_INVENTORY_PATH = PRESSURE_CONTEXT_EXPORT_DIR / "cleaned_k2_large_scale_pressure_plot_inventory.csv"

PRIMARY_CLUSTER_COLUMN = "cleaned_cluster_ward_2"
EXPLORATORY_CLUSTER_COLUMN = "cleaned_cluster_ward_3"
PRIMARY_CLUSTER_A_ID = 1
PRIMARY_CLUSTER_B_ID = 2
ERA5_TIME_CHUNK = 48
CHECKPOINT_EVERY_EVENTS = 10
FORCE_REBUILD_PRESSURE_CONTEXT = False
SAVE_PLOTS = True

MSL_VARIABLE = "mean_sea_level_pressure"
PRESSURE_PLOT_DOMAIN = BoundingBox(lon_min=90.0, lon_max=220.0, lat_min=20.0, lat_max=75.0)
SIBERIAN_HIGH_SEARCH_BOX = BoundingBox(lon_min=100.0, lon_max=145.0, lat_min=45.0, lat_max=72.0)
ALEUTIAN_LOW_SEARCH_BOX = BoundingBox(lon_min=160.0, lon_max=220.0, lat_min=35.0, lat_max=65.0)
RIDGE_EXTENSION_BOX = BoundingBox(lon_min=105.0, lon_max=135.0, lat_min=25.0, lat_max=45.0)

HIGH_EAST_THRESHOLD = 130.0
HIGH_SOUTH_THRESHOLD = 55.0
LOW_EAST_THRESHOLD = 190.0
LOW_SOUTH_THRESHOLD = 50.0

FIELD_NAME = "msl_anomaly_peak_hpa"
FIELD_LABEL = "Mean sea-level pressure anomaly at event peak"
FIELD_UNITS = "hPa"


def maybe_copy_to_drive(path: Path, *, verbose: bool = True):
    if not PERSIST_OUTPUTS_TO_DRIVE:
        return None
    drive_path = Path(DRIVE_OUTPUT_DIR) / path.name
    if path.is_file():
        shutil.copy2(path, drive_path)
        if verbose:
            print("Copied to Drive:", drive_path)
        return drive_path
    return None


def restore_from_drive_cache(path: Path) -> bool:
    if not PERSIST_OUTPUTS_TO_DRIVE:
        return False
    drive_path = Path(DRIVE_OUTPUT_DIR) / path.name
    if not drive_path.exists():
        return False
    path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(drive_path, path)
    print("Restored from Drive:", drive_path, "->", path)
    return True


def ordinal_word(value: int) -> str:
    lookup = {1: "first", 2: "second", 3: "third", 4: "fourth", 5: "fifth"}
    return lookup.get(value, f"{value}th")


def size_rank_descriptor(rank: int, total: int) -> str:
    if total <= 1:
        return "only subgroup"
    if rank == 1:
        return "largest subgroup"
    if rank == total:
        return "smallest subgroup"
    return f"{ordinal_word(rank)}-largest subgroup"


def build_cluster_labels_from_counts(cluster_counts: pd.Series | dict[int, int]):
    counts_dict = {int(cluster_id): int(n_events) for cluster_id, n_events in dict(cluster_counts).items()}
    ranked = sorted(counts_dict.items(), key=lambda item: (-item[1], item[0]))
    rank_lookup = {cluster_id: rank for rank, (cluster_id, _) in enumerate(ranked, start=1)}
    total = len(ranked)
    long_labels = {}
    rows = []
    for cluster_id, n_events in sorted(counts_dict.items()):
        descriptor = size_rank_descriptor(rank_lookup[cluster_id], total)
        long_labels[cluster_id] = f"Cluster {cluster_id}: n={n_events} ({descriptor})"
        rows.append(
            {
                "cluster_id": cluster_id,
                "n_events": n_events,
                "size_rank": rank_lookup[cluster_id],
                "size_descriptor": descriptor,
                "cluster_label": long_labels[cluster_id],
            }
        )
    return long_labels, pd.DataFrame(rows)


def load_cached_dataset(path: Path):
    if not path.exists():
        return None
    with xr.open_dataset(path) as ds:
        return ds.load()


def write_checkpoint_dataset(ds: xr.Dataset, path: Path):
    ds.sortby("event_index").to_netcdf(path)
    maybe_copy_to_drive(path)


def safe_concat(existing_ds: xr.Dataset | None, new_ds: xr.Dataset) -> xr.Dataset:
    if existing_ds is None:
        return new_ds
    return xr.concat([existing_ds, new_ds], dim="event_index").sortby("event_index")


def attach_event_metadata(field: xr.DataArray, row: pd.Series) -> xr.Dataset:
    event_index = int(row["event_index"])
    event_peak = np.datetime64(pd.Timestamp(row["event_peak"]).to_datetime64())
    expanded = field.expand_dims(event_index=[event_index]).to_dataset(name=FIELD_NAME)
    expanded = expanded.assign_coords(
        event_peak=("event_index", [event_peak]),
        cleaned_cluster_k2=("event_index", [int(row[PRIMARY_CLUSTER_COLUMN])]),
    )
    return expanded


def compute_monthly_msl_climatology(ds: xr.Dataset, *, years, months, domain) -> xr.DataArray:
    monthly_sums = {}
    monthly_counts = {}
    for month in months:
        month_sum = None
        month_count = 0
        for year in years:
            start = f"{year}-{month:02d}-01"
            end = (pd.Timestamp(start) + pd.offsets.MonthEnd(1)).strftime("%Y-%m-%dT23:00:00")
            print(f"Climatology month {month:02d}: loading {year}-{month:02d}")
            subset = ds[[MSL_VARIABLE]].sel(
                time=slice(start, end),
                longitude=slice(domain.lon_min, domain.lon_max),
                latitude=slice(domain.lat_max, domain.lat_min),
            )
            msl_hpa = (subset[MSL_VARIABLE] / 100.0).rename("msl_hpa")
            window_sum = msl_hpa.sum("time").load()
            window_count = int(msl_hpa.sizes["time"])
            month_sum = window_sum if month_sum is None else (month_sum + window_sum)
            month_count += window_count
        if month_sum is None or month_count == 0:
            continue
        monthly_sums[int(month)] = month_sum
        monthly_counts[int(month)] = month_count
    climatology_slices = []
    for month in sorted(monthly_sums):
        month_mean = (monthly_sums[month] / monthly_counts[month]).expand_dims(month=[month])
        climatology_slices.append(month_mean)
    climatology = xr.concat(climatology_slices, dim="month").rename("monthly_msl_climatology_hpa")
    climatology.attrs["units"] = "hPa"
    climatology.attrs["source_variable"] = MSL_VARIABLE
    return climatology


def load_or_update_msl_climatology(path: Path, *, years, months, domain, current_ds=None):
    if path.exists():
        climatology = xr.open_dataarray(path).load()
        cached_months = {int(month_value) for month_value in climatology["month"].values.tolist()}
    else:
        climatology = None
        cached_months = set()
    missing_months = sorted(set(months) - cached_months)
    if missing_months:
        print(f"Cached MSL climatology missing months: {missing_months}")
        print("Computing missing MSL climatology months one at a time and checkpointing each completed month to Drive.")
        current_ds = open_arco_era5(chunks={"time": ERA5_TIME_CHUNK}) if current_ds is None else current_ds
        for month in missing_months:
            month_climatology = compute_monthly_msl_climatology(current_ds, years=years, months=[month], domain=domain)
            climatology = month_climatology if climatology is None else xr.concat([climatology, month_climatology], dim="month").sortby("month")
            climatology.to_netcdf(path)
            maybe_copy_to_drive(path)
            print(f"Checkpointed MSL climatology after month {month:02d}")
        climatology_source = "month-by-month checkpoints"
    else:
        climatology_source = "restored cached climatology"
    if climatology is None:
        raise RuntimeError("Unable to load or compute the MSL climatology.")
    return climatology, climatology_source, current_ds


def _subset_box(field: xr.DataArray, box) -> xr.DataArray:
    return field.sel(longitude=slice(box.lon_min, box.lon_max), latitude=slice(box.lat_max, box.lat_min))


def find_box_extremum(field: xr.DataArray, box, *, mode: str) -> dict[str, float]:
    subset = _subset_box(field, box)
    values = np.asarray(subset.values, dtype=float)
    finite_mask = np.isfinite(values)
    if values.size == 0 or not finite_mask.any():
        return {"center_lon_degE": np.nan, "center_lat_degN": np.nan, "extremum_value_hpa": np.nan, "box_mean_hpa": np.nan}
    arg_idx = np.unravel_index(np.nanargmax(values) if mode == "max" else np.nanargmin(values), values.shape)
    lat_idx, lon_idx = int(arg_idx[0]), int(arg_idx[1])
    return {
        "center_lon_degE": float(subset.longitude.values[lon_idx]),
        "center_lat_degN": float(subset.latitude.values[lat_idx]),
        "extremum_value_hpa": float(values[lat_idx, lon_idx]),
        "box_mean_hpa": float(np.nanmean(values)),
    }


def ridge_extension_metrics(field: xr.DataArray, box) -> dict[str, float]:
    subset = _subset_box(field, box)
    values = np.asarray(subset.values, dtype=float)
    if values.size == 0 or not np.isfinite(values).any():
        return {"ridge_box_mean_msl_anomaly_hpa": np.nan, "ridge_box_max_msl_anomaly_hpa": np.nan}
    return {
        "ridge_box_mean_msl_anomaly_hpa": float(np.nanmean(values)),
        "ridge_box_max_msl_anomaly_hpa": float(np.nanmax(values)),
    }


def categorize_position(lon_degE: float, lat_degN: float, *, east_threshold: float, south_threshold: float) -> dict[str, object]:
    east_flag = bool(np.isfinite(lon_degE) and lon_degE >= east_threshold)
    south_flag = bool(np.isfinite(lat_degN) and lat_degN < south_threshold)
    return {"east_flag": east_flag, "south_flag": south_flag, "category": f"{'east' if east_flag else 'west'}_{'south' if south_flag else 'north'}"}


def summarize_pressure_metrics(event_metrics_df: pd.DataFrame, cluster_label_lookup: dict[int, str], cluster_mean_centers_df: pd.DataFrame):
    cluster_summary_rows = []
    for cluster_id, subset in event_metrics_df.groupby(PRIMARY_CLUSTER_COLUMN):
        cluster_id = int(cluster_id)
        center_row = cluster_mean_centers_df.loc[cluster_mean_centers_df["cluster_id"] == cluster_id].iloc[0]
        cluster_summary_rows.append({
            "cluster_id": cluster_id,
            "cluster_label": cluster_label_lookup[cluster_id],
            "n_events": len(subset),
            "mean_high_center_lon_degE": float(subset["high_center_lon_degE"].mean()),
            "mean_high_center_lat_degN": float(subset["high_center_lat_degN"].mean()),
            "mean_high_max_msl_anomaly_hpa": float(subset["high_max_msl_anomaly_hpa"].mean()),
            "mean_low_center_lon_degE": float(subset["low_center_lon_degE"].mean()),
            "mean_low_center_lat_degN": float(subset["low_center_lat_degN"].mean()),
            "mean_low_min_msl_anomaly_hpa": float(subset["low_min_msl_anomaly_hpa"].mean()),
            "mean_pressure_dipole_index_hpa": float(subset["pressure_dipole_index_hpa"].mean()),
            "mean_ridge_box_mean_msl_anomaly_hpa": float(subset["ridge_box_mean_msl_anomaly_hpa"].mean()),
            "percent_high_center_east_of_130E": float(100.0 * subset["high_center_east_of_130E"].mean()),
            "percent_high_center_south_of_55N": float(100.0 * subset["high_center_south_of_55N"].mean()),
            "percent_low_center_east_of_190E": float(100.0 * subset["low_center_east_of_190E"].mean()),
            "percent_low_center_south_of_50N": float(100.0 * subset["low_center_south_of_50N"].mean()),
            "composite_high_center_lon_degE": float(center_row["composite_high_center_lon_degE"]),
            "composite_high_center_lat_degN": float(center_row["composite_high_center_lat_degN"]),
            "composite_high_max_msl_anomaly_hpa": float(center_row["composite_high_max_msl_anomaly_hpa"]),
            "composite_low_center_lon_degE": float(center_row["composite_low_center_lon_degE"]),
            "composite_low_center_lat_degN": float(center_row["composite_low_center_lat_degN"]),
            "composite_low_min_msl_anomaly_hpa": float(center_row["composite_low_min_msl_anomaly_hpa"]),
        })
    cluster_summary_df = pd.DataFrame(cluster_summary_rows)
    category_rows = []
    category_order = ["west_north", "east_north", "west_south", "east_south"]
    for system_name, column_name in [("siberian_high", "high_position_category"), ("aleutian_low", "low_position_category")]:
        count_table = pd.crosstab(event_metrics_df[PRIMARY_CLUSTER_COLUMN], event_metrics_df[column_name])
        count_table = count_table.reindex(index=sorted(cluster_label_lookup), columns=category_order, fill_value=0)
        percent_table = count_table.div(count_table.sum(axis=1), axis=0).fillna(0.0) * 100.0
        for cluster_id in count_table.index:
            for category in count_table.columns:
                category_rows.append({
                    "system": system_name,
                    "cluster_id": int(cluster_id),
                    "cluster_label": cluster_label_lookup[int(cluster_id)],
                    "position_category": category,
                    "count": int(count_table.loc[cluster_id, category]),
                    "percent_of_cluster": float(percent_table.loc[cluster_id, category]),
                })
    position_category_df = pd.DataFrame(category_rows)
    quartile_summary_rows = []
    cluster1_subset_df = event_metrics_df.loc[event_metrics_df["cluster1_quartile_subset"].isin(["lower", "middle", "upper"])].copy()
    if not cluster1_subset_df.empty:
        for subset_name, subset in cluster1_subset_df.groupby("cluster1_quartile_subset"):
            quartile_summary_rows.append({
                "cluster1_quartile_subset": subset_name,
                "n_events": len(subset),
                "mean_high_center_lon_degE": float(subset["high_center_lon_degE"].mean()),
                "mean_high_center_lat_degN": float(subset["high_center_lat_degN"].mean()),
                "mean_high_max_msl_anomaly_hpa": float(subset["high_max_msl_anomaly_hpa"].mean()),
                "mean_low_center_lon_degE": float(subset["low_center_lon_degE"].mean()),
                "mean_low_center_lat_degN": float(subset["low_center_lat_degN"].mean()),
                "mean_low_min_msl_anomaly_hpa": float(subset["low_min_msl_anomaly_hpa"].mean()),
                "mean_pressure_dipole_index_hpa": float(subset["pressure_dipole_index_hpa"].mean()),
                "mean_ridge_box_mean_msl_anomaly_hpa": float(subset["ridge_box_mean_msl_anomaly_hpa"].mean()),
            })
    quartile_summary_df = pd.DataFrame(quartile_summary_rows)
    return cluster_summary_df, position_category_df, quartile_summary_df


def draw_box(ax, box, *, edgecolor, linewidth=1.0, linestyle="--"):
    ax.add_patch(Rectangle((box.lon_min, box.lat_min), box.lon_max - box.lon_min, box.lat_max - box.lat_min, fill=False, edgecolor=edgecolor, linewidth=linewidth, linestyle=linestyle, transform=ccrs.PlateCarree(), zorder=10))


def add_map_features(ax, domain, *, title: str):
    ax.set_extent([domain.lon_min, domain.lon_max, domain.lat_min, domain.lat_max], crs=ccrs.PlateCarree())
    ax.coastlines(resolution="50m", linewidth=0.7)
    ax.add_feature(cfeature.BORDERS.with_scale("50m"), linewidth=0.35, edgecolor="dimgray")
    gl = ax.gridlines(draw_labels=True, linewidth=0.3, color="gray", alpha=0.5, linestyle="--")
    gl.top_labels = False
    gl.right_labels = False
    gl.xlabel_style = {"size": 8.0}
    gl.ylabel_style = {"size": 8.0}
    ax.set_title(title, fontsize=9.8)
    draw_box(ax, SIBERIAN_HIGH_SEARCH_BOX, edgecolor="#1f77b4")
    draw_box(ax, ALEUTIAN_LOW_SEARCH_BOX, edgecolor="#d95f02")
    draw_box(ax, RIDGE_EXTENSION_BOX, edgecolor="#2ca02c", linestyle=":")


def build_symmetric_levels(*arrays, n_levels: int = 13) -> np.ndarray:
    max_abs = 0.0
    for arr in arrays:
        values = np.asarray(arr.values if hasattr(arr, "values") else arr, dtype=float)
        if np.isfinite(values).any():
            max_abs = max(max_abs, float(np.nanmax(np.abs(values))))
    max_abs = 1.0 if (not np.isfinite(max_abs) or max_abs == 0.0) else max_abs
    return np.linspace(-max_abs, max_abs, n_levels)


def plot_cluster_mean_maps(cluster_composite_ds: xr.Dataset, cluster_label_lookup: dict[int, str], cluster_event_counts: dict[int, int], cluster_mean_centers_df: pd.DataFrame):
    cluster_ids = sorted(cluster_event_counts)
    fields = [cluster_composite_ds["msl_anomaly_mean_hpa"].sel(cluster_id=cluster_id) for cluster_id in cluster_ids]
    levels = build_symmetric_levels(*fields)
    fig, axes = plt.subplots(1, len(cluster_ids), figsize=(6.5 * len(cluster_ids), 5.9), subplot_kw={"projection": ccrs.PlateCarree()})
    if len(cluster_ids) == 1:
        axes = [axes]
    mappable = None
    for ax, cluster_id, field in zip(axes, cluster_ids, fields):
        mappable = ax.contourf(field.longitude, field.latitude, field, levels=levels, cmap="RdBu_r", extend="both", transform=ccrs.PlateCarree())
        add_map_features(ax, PRESSURE_PLOT_DOMAIN, title=f"Cluster {cluster_id}\n{cluster_label_lookup[cluster_id]}")
        center_row = cluster_mean_centers_df.loc[cluster_mean_centers_df["cluster_id"] == cluster_id].iloc[0]
        ax.scatter(center_row["composite_high_center_lon_degE"], center_row["composite_high_center_lat_degN"], s=80, marker='*', color="#1f77b4", edgecolor="white", linewidth=0.6, transform=ccrs.PlateCarree(), zorder=11)
        ax.text(center_row["composite_high_center_lon_degE"] + 1.2, center_row["composite_high_center_lat_degN"] + 0.5, "H", color="#1f77b4", fontsize=11, weight="bold", transform=ccrs.PlateCarree(), zorder=11)
        ax.scatter(center_row["composite_low_center_lon_degE"], center_row["composite_low_center_lat_degN"], s=80, marker='v', color="#d95f02", edgecolor="white", linewidth=0.6, transform=ccrs.PlateCarree(), zorder=11)
        ax.text(center_row["composite_low_center_lon_degE"] + 1.2, center_row["composite_low_center_lat_degN"] + 0.5, "L", color="#d95f02", fontsize=11, weight="bold", transform=ccrs.PlateCarree(), zorder=11)
    fig.suptitle("Large-scale pressure context at event peak\nMean sea-level pressure anomaly relative to the NDJF monthly climatology", fontsize=12.5, y=0.98)
    fig.subplots_adjust(top=0.84, bottom=0.18, left=0.05, right=0.98, wspace=0.10)
    cax = fig.add_axes([0.18, 0.08, 0.64, 0.04])
    fig.colorbar(mappable, cax=cax, orientation="horizontal").set_label("MSL anomaly [hPa]")
    return fig


def plot_center_scatter(event_metrics_df: pd.DataFrame, cluster_label_lookup: dict[int, str]):
    cluster_colors = {1: "#1f77b4", 2: "#d95f02", 3: "#2ca02c"}
    fig, ax = plt.subplots(figsize=(9.0, 6.4), subplot_kw={"projection": ccrs.PlateCarree()})
    add_map_features(ax, PRESSURE_PLOT_DOMAIN, title="Event-level Siberian High and Aleutian Low centers")
    for cluster_id in sorted(event_metrics_df[PRIMARY_CLUSTER_COLUMN].dropna().astype(int).unique()):
        subset = event_metrics_df.loc[event_metrics_df[PRIMARY_CLUSTER_COLUMN].astype(int) == cluster_id].copy()
        ax.scatter(subset["high_center_lon_degE"], subset["high_center_lat_degN"], s=26, alpha=0.75, color=cluster_colors.get(cluster_id, "gray"), marker='*', label=f"Cluster {cluster_id} high centers", transform=ccrs.PlateCarree())
        ax.scatter(subset["low_center_lon_degE"], subset["low_center_lat_degN"], s=24, alpha=0.65, color=cluster_colors.get(cluster_id, "gray"), marker='v', transform=ccrs.PlateCarree())
    ax.legend(loc="lower left", frameon=True, fontsize=8)
    return fig


def plot_metric_boxplots(event_metrics_df: pd.DataFrame):
    cluster_ids = sorted(event_metrics_df[PRIMARY_CLUSTER_COLUMN].dropna().astype(int).unique())
    metrics = [("high_center_lon_degE", "High-center lon [degE]"), ("high_center_lat_degN", "High-center lat [degN]"), ("high_max_msl_anomaly_hpa", "High max anomaly [hPa]"), ("low_center_lon_degE", "Low-center lon [degE]"), ("low_center_lat_degN", "Low-center lat [degN]"), ("low_min_msl_anomaly_hpa", "Low min anomaly [hPa]"), ("pressure_dipole_index_hpa", "High-low dipole index [hPa]"), ("ridge_box_mean_msl_anomaly_hpa", "Ridge-box mean anomaly [hPa]")]
    fig, axes = plt.subplots(2, 4, figsize=(15.5, 7.6), constrained_layout=True)
    for ax, (column_name, label) in zip(axes.flat, metrics):
        data = [event_metrics_df.loc[event_metrics_df[PRIMARY_CLUSTER_COLUMN].astype(int) == cluster_id, column_name].dropna().values for cluster_id in cluster_ids]
        box = ax.boxplot(data, showfliers=False, patch_artist=True)
        for patch in box["boxes"]:
            patch.set_facecolor("#d9d9d9")
        ax.set_xticks(np.arange(1, len(cluster_ids) + 1))
        ax.set_xticklabels([f"C{cluster_id}" for cluster_id in cluster_ids])
        ax.set_ylabel(label)
        ax.grid(axis="y", alpha=0.25)
    fig.suptitle("Large-scale pressure metrics by cleaned cluster", fontsize=12.5)
    return fig


def plot_position_category_bars(position_category_df: pd.DataFrame):
    category_order = ["west_north", "east_north", "west_south", "east_south"]
    label_lookup = {"west_north": "W / N", "east_north": "E / N", "west_south": "W / S", "east_south": "E / S"}
    fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.8), constrained_layout=True)
    for ax, system_name, title in zip(axes, ["siberian_high", "aleutian_low"], ["Siberian High center sector", "Aleutian Low center sector"]):
        subset_df = position_category_df.loc[position_category_df["system"] == system_name].copy()
        cluster_ids = sorted(subset_df["cluster_id"].unique().tolist())
        x = np.arange(len(category_order))
        width = 0.35 if len(cluster_ids) == 2 else 0.25
        for offset_index, cluster_id in enumerate(cluster_ids):
            one_cluster = subset_df.loc[subset_df["cluster_id"] == cluster_id].set_index("position_category").reindex(category_order)
            offset = (offset_index - (len(cluster_ids) - 1) / 2.0) * width
            ax.bar(x + offset, one_cluster["percent_of_cluster"].values, width=width, label=f"Cluster {cluster_id}")
        ax.set_xticks(x)
        ax.set_xticklabels([label_lookup[label] for label in category_order])
        ax.set_ylabel("Percent of cluster events [%]")
        ax.set_title(title)
        ax.grid(axis="y", alpha=0.25)
        ax.legend(frameon=False)
    return fig


def plot_quartile_pressure_context(quartile_pressure_summary_df: pd.DataFrame):
    if quartile_pressure_summary_df.empty:
        return None
    subset_order = ["lower", "middle", "upper"]
    plot_df = quartile_pressure_summary_df.set_index("cluster1_quartile_subset").reindex(subset_order).reset_index()
    metrics = [("mean_high_max_msl_anomaly_hpa", "High max anomaly [hPa]"), ("mean_low_min_msl_anomaly_hpa", "Low min anomaly [hPa]"), ("mean_pressure_dipole_index_hpa", "High-low dipole index [hPa]"), ("mean_ridge_box_mean_msl_anomaly_hpa", "Ridge-box mean anomaly [hPa]")]
    fig, axes = plt.subplots(1, 4, figsize=(15.5, 4.4), constrained_layout=True)
    x = np.arange(len(subset_order))
    labels = [name.title() for name in subset_order]
    for ax, (column_name, title) in zip(axes, metrics):
        ax.bar(x, plot_df[column_name].values, color=["#74add1", "#bdbdbd", "#f46d43"])
        ax.set_xticks(x)
        ax.set_xticklabels(labels)
        ax.set_title(title)
        ax.grid(axis="y", alpha=0.25)
    fig.suptitle("Cluster 1 quartile overlay: large-scale pressure context", fontsize=12.5)
    return fig


In [ ]:
paths_to_restore = [CLEANED_CLUSTERED_EVENTS_PATH, NOTEBOOK15_SOLUTION_SUMMARY_PATH, QUARTILE_SELECTION_PATH, MSL_CLIMATOLOGY_PATH]
for path in paths_to_restore:
    if not path.exists():
        restore_from_drive_cache(path)
if not CLEANED_CLUSTERED_EVENTS_PATH.exists():
    raise RuntimeError("Run Notebook 15 first so the cleaned clustered event table exists.")
clustered_df = pd.read_csv(CLEANED_CLUSTERED_EVENTS_PATH)
clustered_df = add_japan_local_time_columns(clustered_df)
clustered_df = clustered_df.reset_index().rename(columns={"index": "event_index"})
required_columns = [PRIMARY_CLUSTER_COLUMN, EXPLORATORY_CLUSTER_COLUMN, "event_peak", "event_peak_jst", "duration_hours", "event_index"]
missing_columns = [column for column in required_columns if column not in clustered_df.columns]
if missing_columns:
    raise RuntimeError(f"Missing required columns in the cleaned clustered event table: {missing_columns}")
cluster_counts = clustered_df[PRIMARY_CLUSTER_COLUMN].dropna().astype(int).value_counts().sort_index()
cluster_label_lookup, cluster_label_df = build_cluster_labels_from_counts(cluster_counts)
clustered_df["cluster1_quartile_subset"] = "not_cluster1"
quartile_overlay_available = False
quartile_subset_counts_df = pd.DataFrame(columns=["cluster1_quartile_subset", "n_events"])
if QUARTILE_SELECTION_PATH.exists():
    quartile_selection_df = pd.read_csv(QUARTILE_SELECTION_PATH)
    if {"event_index", "quartile_subset"}.issubset(quartile_selection_df.columns):
        quartile_lookup = quartile_selection_df.drop_duplicates(subset=["event_index"]).set_index("event_index")["quartile_subset"]
        cluster1_mask = clustered_df[PRIMARY_CLUSTER_COLUMN].astype(int) == PRIMARY_CLUSTER_A_ID
        clustered_df.loc[cluster1_mask, "cluster1_quartile_subset"] = clustered_df.loc[cluster1_mask, "event_index"].map(quartile_lookup).fillna("middle")
        quartile_overlay_available = True
        quartile_subset_counts_df = clustered_df.loc[cluster1_mask, "cluster1_quartile_subset"].value_counts().rename_axis("cluster1_quartile_subset").reset_index(name="n_events").sort_values("cluster1_quartile_subset").reset_index(drop=True)
climatology_years = sorted(pd.to_datetime(clustered_df["event_peak"]).dt.year.unique().tolist())
required_months = sorted(pd.to_datetime(clustered_df["event_peak"]).dt.month.unique().tolist())
era5_runtime_ds = None
msl_climatology, msl_climatology_source, era5_runtime_ds = load_or_update_msl_climatology(MSL_CLIMATOLOGY_PATH, years=climatology_years, months=required_months, domain=PRESSURE_PLOT_DOMAIN, current_ds=era5_runtime_ds)
climatology_summary_df = pd.DataFrame({"metric": ["msl climatology source", "event years represented", "required months", "plot domain", "siberian high search box", "aleutian low search box", "ridge-extension box", "high east/south thresholds [degE, degN]", "low east/south thresholds [degE, degN]", "quartile overlay available"], "value": [msl_climatology_source, f"{min(climatology_years)}-{max(climatology_years)} ({len(climatology_years)} years)", ", ".join(str(month) for month in required_months), f"{PRESSURE_PLOT_DOMAIN.lon_min}-{PRESSURE_PLOT_DOMAIN.lon_max}E, {PRESSURE_PLOT_DOMAIN.lat_min}-{PRESSURE_PLOT_DOMAIN.lat_max}N", f"{SIBERIAN_HIGH_SEARCH_BOX.lon_min}-{SIBERIAN_HIGH_SEARCH_BOX.lon_max}E, {SIBERIAN_HIGH_SEARCH_BOX.lat_min}-{SIBERIAN_HIGH_SEARCH_BOX.lat_max}N", f"{ALEUTIAN_LOW_SEARCH_BOX.lon_min}-{ALEUTIAN_LOW_SEARCH_BOX.lon_max}E, {ALEUTIAN_LOW_SEARCH_BOX.lat_min}-{ALEUTIAN_LOW_SEARCH_BOX.lat_max}N", f"{RIDGE_EXTENSION_BOX.lon_min}-{RIDGE_EXTENSION_BOX.lon_max}E, {RIDGE_EXTENSION_BOX.lat_min}-{RIDGE_EXTENSION_BOX.lat_max}N", f"{HIGH_EAST_THRESHOLD}, {HIGH_SOUTH_THRESHOLD}", f"{LOW_EAST_THRESHOLD}, {LOW_SOUTH_THRESHOLD}", quartile_overlay_available]})
climatology_month_means_df = pd.DataFrame({"month": [int(month_value) for month_value in msl_climatology["month"].values.tolist()], "domain_mean_hpa": [round(float(msl_climatology.sel(month=month_value).mean().values), 3) for month_value in msl_climatology["month"].values.tolist()]})
print("Cleaned cluster labels available for the large-scale pressure-context analysis")
display(cluster_label_df)
print("\nMSL climatology context")
display(climatology_summary_df)
print("\nMSL climatology domain means by month")
display(climatology_month_means_df)
if quartile_overlay_available:
    print("\nCluster 1 quartile-subset counts")
    display(quartile_subset_counts_df)


In [ ]:
required_globals = ["clustered_df", "msl_climatology", "find_box_extremum", "ridge_extension_metrics", "categorize_position"]
missing_globals = [name for name in required_globals if name not in globals()]
if missing_globals:
    raise RuntimeError("Run the setup/import cell and the Notebook 20 context/climatology cell before the one-event sanity-check cell. " f"Missing globals: {missing_globals}")
if "era5_runtime_ds" not in globals() or era5_runtime_ds is None:
    era5_runtime_ds = open_arco_era5(chunks={"time": ERA5_TIME_CHUNK})
sample_event_row = clustered_df.iloc[0].copy()
sample_peak = pd.Timestamp(sample_event_row["event_peak"])
sample_snapshot = load_snapshot(era5_runtime_ds, sample_peak, variables=[MSL_VARIABLE], domain=PRESSURE_PLOT_DOMAIN, level=None)
sample_msl_hpa = (sample_snapshot[MSL_VARIABLE] / 100.0).rename("msl_hpa_peak")
sample_anomaly = (sample_msl_hpa - msl_climatology.sel(month=sample_peak.month)).rename(FIELD_NAME)
sample_high = find_box_extremum(sample_anomaly, SIBERIAN_HIGH_SEARCH_BOX, mode="max")
sample_low = find_box_extremum(sample_anomaly, ALEUTIAN_LOW_SEARCH_BOX, mode="min")
sample_ridge = ridge_extension_metrics(sample_anomaly, RIDGE_EXTENSION_BOX)
sample_high_pos = categorize_position(sample_high["center_lon_degE"], sample_high["center_lat_degN"], east_threshold=HIGH_EAST_THRESHOLD, south_threshold=HIGH_SOUTH_THRESHOLD)
sample_low_pos = categorize_position(sample_low["center_lon_degE"], sample_low["center_lat_degN"], east_threshold=LOW_EAST_THRESHOLD, south_threshold=LOW_SOUTH_THRESHOLD)
sample_summary_df = pd.DataFrame({"metric": ["sample event peak", "sample cluster", "sample cluster1 quartile subset", "sample high-center longitude [degE]", "sample high-center latitude [degN]", "sample high max MSL anomaly [hPa]", "sample low-center longitude [degE]", "sample low-center latitude [degN]", "sample low min MSL anomaly [hPa]", "sample ridge-box mean anomaly [hPa]", "sample ridge-box max anomaly [hPa]", "sample pressure dipole index [hPa]", "sample high-position category", "sample low-position category"], "value": [str(sample_peak), int(sample_event_row[PRIMARY_CLUSTER_COLUMN]), sample_event_row["cluster1_quartile_subset"], round(sample_high["center_lon_degE"], 3), round(sample_high["center_lat_degN"], 3), round(sample_high["extremum_value_hpa"], 3), round(sample_low["center_lon_degE"], 3), round(sample_low["center_lat_degN"], 3), round(sample_low["extremum_value_hpa"], 3), round(sample_ridge["ridge_box_mean_msl_anomaly_hpa"], 3), round(sample_ridge["ridge_box_max_msl_anomaly_hpa"], 3), round(sample_high["extremum_value_hpa"] - sample_low["extremum_value_hpa"], 3), sample_high_pos["category"], sample_low_pos["category"]]})
print("One-event large-scale pressure-context sanity check before the full event loop")
display(sample_summary_df)
if not np.isfinite(sample_high["extremum_value_hpa"]) or not np.isfinite(sample_low["extremum_value_hpa"]):
    raise RuntimeError("The sanity-check event returned a missing Siberian High or Aleutian Low extremum in the search boxes. Do not run the full event loop until the MSL field or search boxes are corrected.")


In [ ]:
required_globals = ["clustered_df", "cluster_label_lookup", "find_box_extremum", "ridge_extension_metrics", "categorize_position", "load_cached_dataset", "write_checkpoint_dataset", "safe_concat", "attach_event_metadata", "summarize_pressure_metrics"]
missing_globals = [name for name in required_globals if name not in globals()]
if missing_globals:
    raise RuntimeError("Run the setup/import cell, the Notebook 20 context/climatology cell, and the one-event sanity-check cell before the full analysis cell. " f"Missing globals: {missing_globals}")
for path in [EVENT_STACK_PATH, EVENT_METRICS_PATH, CLUSTER_COMPOSITE_PATH, CLUSTER_SUMMARY_PATH, POSITION_CATEGORY_PATH, QUARTILE_PRESSURE_SUMMARY_PATH]:
    if not path.exists():
        restore_from_drive_cache(path)
reuse_final_outputs = ((not FORCE_REBUILD_PRESSURE_CONTEXT) and EVENT_STACK_PATH.exists() and EVENT_METRICS_PATH.exists() and CLUSTER_COMPOSITE_PATH.exists() and CLUSTER_SUMMARY_PATH.exists() and POSITION_CATEGORY_PATH.exists())
if reuse_final_outputs:
    event_stack_ds = load_cached_dataset(EVENT_STACK_PATH)
    event_metrics_df = pd.read_csv(EVENT_METRICS_PATH)
    cluster_composite_ds = load_cached_dataset(CLUSTER_COMPOSITE_PATH)
    cluster_summary_df = pd.read_csv(CLUSTER_SUMMARY_PATH)
    position_category_df = pd.read_csv(POSITION_CATEGORY_PATH)
    quartile_pressure_summary_df = pd.read_csv(QUARTILE_PRESSURE_SUMMARY_PATH) if QUARTILE_PRESSURE_SUMMARY_PATH.exists() else pd.DataFrame()
else:
    if not FORCE_REBUILD_PRESSURE_CONTEXT:
        if not EVENT_STACK_PARTIAL_PATH.exists():
            restore_from_drive_cache(EVENT_STACK_PARTIAL_PATH)
        if not EVENT_METRICS_PARTIAL_PATH.exists():
            restore_from_drive_cache(EVENT_METRICS_PARTIAL_PATH)
    partial_metrics_df = pd.read_csv(EVENT_METRICS_PARTIAL_PATH) if (EVENT_METRICS_PARTIAL_PATH.exists() and not FORCE_REBUILD_PRESSURE_CONTEXT) else pd.DataFrame()
    partial_stack_ds = load_cached_dataset(EVENT_STACK_PARTIAL_PATH) if (EVENT_STACK_PARTIAL_PATH.exists() and not FORCE_REBUILD_PRESSURE_CONTEXT) else None
    processed_event_indices = set(partial_metrics_df["event_index"].astype(int).tolist()) if not partial_metrics_df.empty else set()
    pending_df = clustered_df.loc[~clustered_df["event_index"].isin(processed_event_indices)].copy().sort_values("event_peak")
    if "era5_runtime_ds" not in globals() or era5_runtime_ds is None:
        era5_runtime_ds = open_arco_era5(chunks={"time": ERA5_TIME_CHUNK})
    new_metric_rows = []
    event_stack_ds = partial_stack_ds
    total_pending = len(pending_df)
    for pending_number, (_, row) in enumerate(pending_df.iterrows(), start=1):
        peak_time = pd.Timestamp(row["event_peak"])
        snapshot = load_snapshot(era5_runtime_ds, peak_time, variables=[MSL_VARIABLE], domain=PRESSURE_PLOT_DOMAIN, level=None)
        msl_hpa = (snapshot[MSL_VARIABLE] / 100.0).rename("msl_hpa_peak")
        anomaly_field = (msl_hpa - msl_climatology.sel(month=peak_time.month)).rename(FIELD_NAME)
        high_metrics = find_box_extremum(anomaly_field, SIBERIAN_HIGH_SEARCH_BOX, mode="max")
        low_metrics = find_box_extremum(anomaly_field, ALEUTIAN_LOW_SEARCH_BOX, mode="min")
        ridge_metrics = ridge_extension_metrics(anomaly_field, RIDGE_EXTENSION_BOX)
        high_position = categorize_position(high_metrics["center_lon_degE"], high_metrics["center_lat_degN"], east_threshold=HIGH_EAST_THRESHOLD, south_threshold=HIGH_SOUTH_THRESHOLD)
        low_position = categorize_position(low_metrics["center_lon_degE"], low_metrics["center_lat_degN"], east_threshold=LOW_EAST_THRESHOLD, south_threshold=LOW_SOUTH_THRESHOLD)
        new_metric_rows.append({"event_index": int(row["event_index"]), "event_peak": peak_time, "event_peak_jst": row["event_peak_jst"], "duration_hours": float(row["duration_hours"]) if pd.notna(row["duration_hours"]) else np.nan, PRIMARY_CLUSTER_COLUMN: int(row[PRIMARY_CLUSTER_COLUMN]), EXPLORATORY_CLUSTER_COLUMN: int(row[EXPLORATORY_CLUSTER_COLUMN]), "cluster1_quartile_subset": row["cluster1_quartile_subset"], "event_month": int(peak_time.month), "high_center_lon_degE": high_metrics["center_lon_degE"], "high_center_lat_degN": high_metrics["center_lat_degN"], "high_max_msl_anomaly_hpa": high_metrics["extremum_value_hpa"], "high_box_mean_msl_anomaly_hpa": high_metrics["box_mean_hpa"], "high_center_east_of_130E": high_position["east_flag"], "high_center_south_of_55N": high_position["south_flag"], "high_position_category": high_position["category"], "low_center_lon_degE": low_metrics["center_lon_degE"], "low_center_lat_degN": low_metrics["center_lat_degN"], "low_min_msl_anomaly_hpa": low_metrics["extremum_value_hpa"], "low_box_mean_msl_anomaly_hpa": low_metrics["box_mean_hpa"], "low_center_east_of_190E": low_position["east_flag"], "low_center_south_of_50N": low_position["south_flag"], "low_position_category": low_position["category"], "pressure_dipole_index_hpa": high_metrics["extremum_value_hpa"] - low_metrics["extremum_value_hpa"], **ridge_metrics})
        event_stack_ds = safe_concat(event_stack_ds, attach_event_metadata(anomaly_field, row))
        if pending_number % CHECKPOINT_EVERY_EVENTS == 0 or pending_number == total_pending:
            combined_metrics_df = pd.concat([partial_metrics_df, pd.DataFrame(new_metric_rows)], ignore_index=True)
            combined_metrics_df = combined_metrics_df.drop_duplicates(subset=["event_index"], keep="last").sort_values("event_index").reset_index(drop=True)
            combined_metrics_df.to_csv(EVENT_METRICS_PARTIAL_PATH, index=False)
            maybe_copy_to_drive(EVENT_METRICS_PARTIAL_PATH)
            write_checkpoint_dataset(event_stack_ds, EVENT_STACK_PARTIAL_PATH)
            print(f"Large-scale pressure-context event processing: {pending_number}/{total_pending} pending events")
    event_metrics_df = pd.concat([partial_metrics_df, pd.DataFrame(new_metric_rows)], ignore_index=True)
    event_metrics_df = event_metrics_df.drop_duplicates(subset=["event_index"], keep="last").sort_values("event_index").reset_index(drop=True)
    write_checkpoint_dataset(event_stack_ds, EVENT_STACK_PATH)
    event_metrics_df.to_csv(EVENT_METRICS_PATH, index=False)
    maybe_copy_to_drive(EVENT_METRICS_PATH)
    cluster_ids = sorted(cluster_label_lookup)
    mean_slices = []
    count_slices = []
    cluster_mean_center_rows = []
    for cluster_id in cluster_ids:
        cluster_event_indices = event_metrics_df.loc[event_metrics_df[PRIMARY_CLUSTER_COLUMN].astype(int) == int(cluster_id), "event_index"].astype(int).tolist()
        cluster_subset = event_stack_ds.sel(event_index=cluster_event_indices)[FIELD_NAME].sortby("event_index")
        cluster_mean_field = cluster_subset.mean(dim="event_index", skipna=True)
        mean_slices.append(cluster_mean_field.expand_dims(cluster_id=[int(cluster_id)]))
        count_slices.append(cluster_subset.count(dim="event_index").astype(np.int32).expand_dims(cluster_id=[int(cluster_id)]))
        high_center = find_box_extremum(cluster_mean_field, SIBERIAN_HIGH_SEARCH_BOX, mode="max")
        low_center = find_box_extremum(cluster_mean_field, ALEUTIAN_LOW_SEARCH_BOX, mode="min")
        cluster_mean_center_rows.append({"cluster_id": int(cluster_id), "composite_high_center_lon_degE": high_center["center_lon_degE"], "composite_high_center_lat_degN": high_center["center_lat_degN"], "composite_high_max_msl_anomaly_hpa": high_center["extremum_value_hpa"], "composite_low_center_lon_degE": low_center["center_lon_degE"], "composite_low_center_lat_degN": low_center["center_lat_degN"], "composite_low_min_msl_anomaly_hpa": low_center["extremum_value_hpa"]})
    cluster_mean_centers_df = pd.DataFrame(cluster_mean_center_rows)
    cluster_composite_ds = xr.Dataset({"msl_anomaly_mean_hpa": xr.concat(mean_slices, dim="cluster_id"), "msl_anomaly_count": xr.concat(count_slices, dim="cluster_id")})
    cluster_composite_ds.to_netcdf(CLUSTER_COMPOSITE_PATH)
    maybe_copy_to_drive(CLUSTER_COMPOSITE_PATH)
    cluster_summary_df, position_category_df, quartile_pressure_summary_df = summarize_pressure_metrics(event_metrics_df, cluster_label_lookup, cluster_mean_centers_df)
    cluster_summary_df.to_csv(CLUSTER_SUMMARY_PATH, index=False)
    position_category_df.to_csv(POSITION_CATEGORY_PATH, index=False)
    maybe_copy_to_drive(CLUSTER_SUMMARY_PATH)
    maybe_copy_to_drive(POSITION_CATEGORY_PATH)
    if not quartile_pressure_summary_df.empty:
        quartile_pressure_summary_df.to_csv(QUARTILE_PRESSURE_SUMMARY_PATH, index=False)
        maybe_copy_to_drive(QUARTILE_PRESSURE_SUMMARY_PATH)
print("Large-scale pressure-context cluster summary")
display(cluster_summary_df)
print("\nLarge-scale pressure position-category summary")
display(position_category_df)
if not quartile_pressure_summary_df.empty:
    print("\nCluster 1 quartile-subset pressure-context summary")
    display(quartile_pressure_summary_df)


In [ ]:
required_globals = ["cluster_composite_ds", "event_metrics_df", "position_category_df", "cluster_label_lookup", "plot_cluster_mean_maps", "plot_center_scatter", "plot_metric_boxplots", "plot_position_category_bars"]
missing_globals = [name for name in required_globals if name not in globals()]
if missing_globals:
    raise RuntimeError("Run the Notebook 20 full analysis cell before the plotting cell. " f"Missing globals: {missing_globals}")
cluster_event_counts = event_metrics_df[PRIMARY_CLUSTER_COLUMN].astype(int).value_counts().sort_index().to_dict()
cluster_mean_centers_df = cluster_summary_df[["cluster_id", "composite_high_center_lon_degE", "composite_high_center_lat_degN", "composite_high_max_msl_anomaly_hpa", "composite_low_center_lon_degE", "composite_low_center_lat_degN", "composite_low_min_msl_anomaly_hpa"]].copy()
plot_records = []
cluster_map_fig = plot_cluster_mean_maps(cluster_composite_ds, cluster_label_lookup, cluster_event_counts, cluster_mean_centers_df)
cluster_map_path = PLOT_DIR / "cleaned_k2_large_scale_pressure_cluster_mean_maps.png"
cluster_map_drive = None
if SAVE_PLOTS:
    cluster_map_fig.savefig(cluster_map_path, dpi=180, bbox_inches="tight")
    cluster_map_drive = maybe_copy_to_drive(cluster_map_path, verbose=False)
plot_records.append({"plot_kind": "cluster_mean_maps", "local_path": str(cluster_map_path), "drive_path": str(cluster_map_drive) if cluster_map_drive else ""})
plt.show(cluster_map_fig)
scatter_fig = plot_center_scatter(event_metrics_df, cluster_label_lookup)
scatter_path = PLOT_DIR / "cleaned_k2_large_scale_pressure_center_scatter.png"
scatter_drive = None
if SAVE_PLOTS:
    scatter_fig.savefig(scatter_path, dpi=180, bbox_inches="tight")
    scatter_drive = maybe_copy_to_drive(scatter_path, verbose=False)
plot_records.append({"plot_kind": "center_scatter", "local_path": str(scatter_path), "drive_path": str(scatter_drive) if scatter_drive else ""})
plt.show(scatter_fig)
metric_boxplot_fig = plot_metric_boxplots(event_metrics_df)
metric_boxplot_path = PLOT_DIR / "cleaned_k2_large_scale_pressure_metric_boxplots.png"
metric_boxplot_drive = None
if SAVE_PLOTS:
    metric_boxplot_fig.savefig(metric_boxplot_path, dpi=180, bbox_inches="tight")
    metric_boxplot_drive = maybe_copy_to_drive(metric_boxplot_path, verbose=False)
plot_records.append({"plot_kind": "metric_boxplots", "local_path": str(metric_boxplot_path), "drive_path": str(metric_boxplot_drive) if metric_boxplot_drive else ""})
plt.show(metric_boxplot_fig)
position_bar_fig = plot_position_category_bars(position_category_df)
position_bar_path = PLOT_DIR / "cleaned_k2_large_scale_pressure_position_category_bars.png"
position_bar_drive = None
if SAVE_PLOTS:
    position_bar_fig.savefig(position_bar_path, dpi=180, bbox_inches="tight")
    position_bar_drive = maybe_copy_to_drive(position_bar_path, verbose=False)
plot_records.append({"plot_kind": "position_category_bars", "local_path": str(position_bar_path), "drive_path": str(position_bar_drive) if position_bar_drive else ""})
plt.show(position_bar_fig)
if 'quartile_pressure_summary_df' in globals() and not quartile_pressure_summary_df.empty:
    quartile_fig = plot_quartile_pressure_context(quartile_pressure_summary_df)
    quartile_path = PLOT_DIR / "cleaned_k2_cluster1_large_scale_pressure_quartile_bars.png"
    quartile_drive = None
    if SAVE_PLOTS and quartile_fig is not None:
        quartile_fig.savefig(quartile_path, dpi=180, bbox_inches="tight")
        quartile_drive = maybe_copy_to_drive(quartile_path, verbose=False)
    plot_records.append({"plot_kind": "cluster1_quartile_pressure_context", "local_path": str(quartile_path), "drive_path": str(quartile_drive) if quartile_drive else ""})
    if quartile_fig is not None:
        plt.show(quartile_fig)
plot_inventory_df = pd.DataFrame(plot_records)
plot_inventory_df.to_csv(PLOT_INVENTORY_PATH, index=False)
maybe_copy_to_drive(PLOT_INVENTORY_PATH)
print("Saved large-scale pressure-context plot inventory")
display(plot_inventory_df)


In [ ]:
required_globals = ["cluster_label_df", "cluster_summary_df", "position_category_df", "quartile_pressure_summary_df", "climatology_summary_df"]
missing_globals = [name for name in required_globals if name not in globals()]
if missing_globals:
    raise RuntimeError("Run the Notebook 20 context, analysis, and plotting cells before the summary cell. " f"Missing globals: {missing_globals}")
print("Notebook 20 large-scale pressure-context summary")
display(climatology_summary_df)
print("\nCleaned cluster counts")
display(cluster_label_df)
print("\nCluster-wise pressure-context summary")
display(cluster_summary_df)
print("\nCluster occurrence by large-scale pressure center sector")
display(position_category_df)
if not quartile_pressure_summary_df.empty:
    print("\nCluster 1 quartile-overlay pressure-context summary")
    display(quartile_pressure_summary_df)
print("\nNotebook 20 now does the following:\n- identifies the Eastern Siberian High center using the maximum positive MSL anomaly inside a broad Siberian search box\n- identifies the Aleutian Low center using the minimum MSL anomaly inside a broad North Pacific search box\n- quantifies southward ridge extension toward East Asia in a separate ridge box\n- compares those large-scale pressure metrics across the cleaned clusters without feeding them back into the clustering")
